# 03. Spec-driven candidate-evidence smoke run

This notebook is the execution-facing successor to the component workbench. Its job is to make every gate of the approved candidate-evidence smoke contract visible before, during, and after the bounded local run. The contract—not this notebook or prior interactive work—is the authority for the candidate envelope.

We will work one gate at a time: contract preflight, validated baseline-cache reuse, the eight declared candidate/seat fast-and-clean-audit pairs, then the immutable packet write. A failed gate stops the run. No cell may add values, widen the suite, change Kaggriculture source, overwrite a packet, or interpret the one-seed result as a ranking.

> **Current stage:** contract preflight only. This notebook has no stored execution output and does not run a candidate merely by being opened.


## 1. Load the approved envelope

The only candidate authority in this notebook is the source-controlled approved contract. We locate the project rather than relying on the notebook kernel's current working directory, then import the reviewed adapter source. Reading a JSON file is not validation, lifecycle authorization, cache reuse, or simulator execution; those are separate gates below.


In [ ]:
from pathlib import Path
import sys


def locate_project_root(start: Path) -> Path:
    """Find this source-controlled project without trusting notebook cwd."""
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / "experiments/schema/experiment-spec-v1.schema.json").is_file():
            return directory
        nested_project = directory / "carrot-tuning-system"
        if (nested_project / "experiments/schema/experiment-spec-v1.schema.json").is_file():
            return nested_project
    raise RuntimeError("Could not locate the carrot-tuning-system project root.")


PROJECT_ROOT = locate_project_root(Path.cwd())
REPOSITORY_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(REPOSITORY_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

from evaluator.runner import LocalInnerLoopAdapter, LocalRunRequest

ADAPTER = LocalInnerLoopAdapter()
SPEC_PATH = (
    PROJECT_ROOT
    / "experiments/approved/carrot_investment/seed_opportunity_candidate_evidence_smoke_v1.json"
)
TARGET_ROOT = REPOSITORY_ROOT.parent / "kaggriculture"
REPORT_DIRECTORY = REPOSITORY_ROOT / "var/carrot-tuning-system/candidate-evidence-smoke"
REQUEST = LocalRunRequest(SPEC_PATH, TARGET_ROOT, REPORT_DIRECTORY)


## 2. Reviewable contract preflight

This function performs only the non-simulator gates: read/schema validation, explicit target fingerprint comparison, lifecycle authorization, and local-control validation. It returns the facts needed by the next stage, but it deliberately does not plan cache paths, call `run()`, replay a candidate, run an audit, create a directory, or write a packet. Keeping this first cell narrow makes a failed approval or target check easy to diagnose.


In [ ]:
def run_contract_preflight(request: LocalRunRequest) -> dict:
    """Return pre-simulation facts for one approved contract."""
    specification = ADAPTER.read_and_validate_spec(request.spec_path)
    binding = ADAPTER.bind_and_validate_target(specification, request.target_root)
    lifecycle = ADAPTER.authorize_lifecycle(specification)
    controls = ADAPTER.validate_local_controls(specification, binding)

    # These values are presentation facts for the next reviewed stage, not evidence.
    return {
        "specification": specification,
        "binding": binding,
        "lifecycle": lifecycle,
        "controls": controls,
    }


# Run this cell only when reviewing the preflight. It cannot call a simulator.
# PREFLIGHT = run_contract_preflight(REQUEST)


In [ ]:
def render_contract_preflight(preflight: dict) -> None:
    """Presentation only: render facts from an already-computed preflight."""
    from IPython.display import Markdown, display

    specification = preflight["specification"]
    lifecycle = preflight["lifecycle"]
    controls = preflight["controls"]
    display(Markdown(
        f"""### Approved contract preflight

| Gate | Result |
|:---|:---|
| Contract | `{specification['experiment_id']}` |
| Lifecycle | **{lifecycle.declared_status}** |
| Target fingerprint | **Passed** |
| Local controls | **{len(controls.checks)} checks passed** |
| Simulator run | No |
| Packet written | No |

> Passing this preflight only permits review of the next cache-reuse stage. It does not itself evaluate a candidate.
"""
    ))


# After computing PREFLIGHT above, render it separately with:
# render_contract_preflight(PREFLIGHT)


## Next reviewed gate

Run and inspect the preflight cells first. If every preflight gate passes, the next notebook section will inspect and validate the two existing baseline caches without materializing, evaluating, or auditing anything. Only after that review will we wire the approved contract through the remaining `run()` orchestration stages.
